# PT-W3-D2 概念实验：Ontology Compiler

把业务语义编译成 Agent 可消费的 Minimum Design Artifact。流水线：Validate → Resolve → Assemble → Emit → Report。

## 实验 1：声明式规则展开为实例级制品

对应 D2 的三原则：业务事实与运行时分离、对象优先、绑定优先。规则声明不写实例 ID，由编译阶段解析。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False

DECLARATIONS = {
    'object_types': {'Lease', 'Space', 'Inspection'},
    'events': {'LeaseTerminated', 'InspectionCompleted'},
    'capabilities': {'release_space', 'create_inspection'},
    'bindings': {'occupies', 'requires'},
}
RULE = {'goal': '租约终止后释放铺位', 'object': 'Lease', 'event': 'LeaseTerminated', 'binding': 'occupies', 'capability': 'release_space', 'guard': 'InspectionCompleted'}
INSTANCES = {'Lease': {'L-001': {'space_id': 'A101', 'state': 'Terminating'}}, 'Space': {'A101': {'state': 'Locked'}}}
print('声明规则:', RULE)

In [ ]:
def compile_rule(rule, instances):
    diagnostics = []
    if rule['object'] not in DECLARATIONS['object_types']: diagnostics.append('未知对象')
    if rule['event'] not in DECLARATIONS['events']: diagnostics.append('未知事件')
    if rule['binding'] not in DECLARATIONS['bindings']: diagnostics.append('未知绑定')
    if rule['capability'] not in DECLARATIONS['capabilities']: diagnostics.append('未知能力')
    if diagnostics:
        return {'status': 'rejected', 'report': diagnostics}
    artifacts = []
    for lease_id, lease in instances['Lease'].items():
        if lease['state'] == 'Terminating':
            artifacts.append({'goal': rule['goal'], 'object_ref': f'Lease:{lease_id}', 'event_ref': rule['event'], 'capability_ref': rule['capability'], 'target_ref': f'Space:{lease["space_id"]}', 'governance': rule['guard']})
    return {'status': 'emitted', 'artifacts': artifacts, 'report': ['全部声明已解析']}

compiled = compile_rule(RULE, INSTANCES)
print('Validate/Resolve/Assemble/Emit/Report =>', compiled)

In [ ]:
bad_rule = dict(RULE, capability='unknown_capability')
rejected = compile_rule(bad_rule, INSTANCES)
print('故意注入缺口:', rejected)

stages = ['Validate', 'Resolve', 'Assemble', 'Emit', 'Report']
values = [1, 1, len(compiled.get('artifacts', [])), len(compiled.get('artifacts', [])), len(compiled['report'])]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(stages, values, marker='o', color='#264653')
ax.set_title('Ontology Compiler 五步流水线'); ax.set_ylabel('记录数'); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()